# 02-19c Local nnU-Net PHE with CT-only pseudo-ICH prior control v2

This is a diagnostic resplit follow-up to `02_19b`. It keeps the same no-teacher CT-only pseudo-ICH control design, but changes the train/val/test assignment to study split sensitivity. It writes a separate experiment/output so the original `02_19`/`02_19b` results remain unchanged.


> Split note: `02_19c` uses a locked resplit for secondary comparison. Report it separately from `02_12`/`02_19`/`02_19b` headline results unless all compared models are rerun on this exact split.

Default profile: `phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval` with brain-window + subdural-window CT input, lightweight skull stripping, CT-only pseudo-ICH prior, fixed 55 HU global candidate branch, CT-derived local Otsu refinement, >80 HU artifact control, and conservative PHE post-processing.

Fairness rule: pseudo-ICH priors are generated from CT intensities only. Manual PHE masks are used only as PHE labels/evaluation targets and optional QC overlays, not to generate prior channels.


In [1]:
from pathlib import Path
import os
import sys
import re
import json
import shutil
import time
import gc

import numpy as np
import pandas as pd

# Local-only setup for this notebook. Keep Kaggle disabled so paths do not drift.
IS_KAGGLE = False
KAGGLE_INPUT = None
WORK_ROOT = Path.cwd().resolve()
LOCAL_ROOT = WORK_ROOT
PROJECT_ROOT = WORK_ROOT

# Local manual paths. Edit these only if your workspace layout changes.
USER_NNUNET_REPO = PROJECT_ROOT / 'nnUNet-master'
USER_PHE_ROOT = PROJECT_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT'
USER_SPLIT_CSV = None    # leave None to use the embedded fair split IDs

# Locked 02_19c resplit. Use this exact split for every model rerun in the 19c comparison table.
# Keep 19c results separate from the original 02_12/02_14/02_19/02_19b benchmark split.
VAL_SCAN_IDS = {'0013', '0014', '0051', '0060', '0061', '0078', '0113', '0116', '0160', '0167'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0080', '0084', '0095', '0096', '0115', '0119', '0130', '0138'}

# 02_19c is a no-teacher control. It does not load 02_13/02_13b checkpoints.
PSEUDO_ICH_SOURCE = 'ct_fixed55_localotsu_no_gt'
PSEUDO_ICH_POLICY = (
    'Pseudo-ICH prior is generated from PHE-SICH CT intensities only: fixed 55 HU global branch, '
    'local Otsu inside a CT-derived high-density ROI, >80 HU artifact control by adjacency to 40-80 HU core, '
    'and morphology/connected-component filtering. No true ICH labels and no PHE masks are used for prior generation.'
)

# Brain CT preprocessing profile from PHE-SICH/PESE-style preprocessing notes.
# L=40, W=120 => HU range [-20, 100]. The raw NIfTI CT is kept for pseudo-ICH HU thresholding.
# nnU-Net CT input channels can use one or more fixed CT windows, all skull-stripped with the same brain mask.
CT_PREPROCESS_PROFILE = 'brain_subdural_window_l40w120_ct55_otsu_skullstrip_v2'
CT_WINDOW_LEVEL_HU = 40.0
CT_WINDOW_WIDTH_HU = 120.0
CT_WINDOW_LOW_HU = CT_WINDOW_LEVEL_HU - CT_WINDOW_WIDTH_HU / 2.0
CT_WINDOW_HIGH_HU = CT_WINDOW_LEVEL_HU + CT_WINDOW_WIDTH_HU / 2.0
CT_WINDOW_OUTPUT_SCALE = 100.0
CT_WINDOW_LIBRARY = {
    'brain': {'level': 40.0, 'width': 120.0},
    'subdural': {'level': 75.0, 'width': 215.0},
    'bone': {'level': 600.0, 'width': 2800.0},
}
CT_SKULL_STRIP_ENABLED = True
CT_SKULL_STRIP_HEAD_LOW_HU = -300.0
CT_SKULL_STRIP_BONE_HU = 150.0
CT_SKULL_STRIP_BONE_DILATE_ITER = 1
CT_SKULL_STRIP_MIN_COMPONENT_VOXELS = 500

EXPERIMENTS = {
    'phe_pseudo_binary_prior': {
        'dataset_id': 190,
        'prior_channels': ['ich_binary'],
        'description': 'CT + pseudo hard ICH mask from CT thresholding',
    },
    'phe_pseudo_probability_prior': {
        'dataset_id': 191,
        'prior_channels': ['ich_probability'],
        'description': 'CT + pseudo soft ICH probability from CT thresholding',
    },
    'phe_pseudo_distance_prior': {
        'dataset_id': 192,
        'prior_channels': ['ich_distance'],
        'description': 'CT + distance-to-pseudo-ICH proximity map',
    },
    'phe_pseudo_prob_distance_prior': {
        'dataset_id': 193,
        'prior_channels': ['ich_probability', 'ich_distance'],
        'description': 'CT + pseudo ICH probability + distance-to-pseudo-ICH proximity',
    },
    'phe_pseudo_prob_distance_dilated_prior': {
        'dataset_id': 194,
        'prior_channels': ['ich_probability', 'ich_distance', 'ich_dilated_roi'],
        'description': 'CT + pseudo ICH probability + distance + dilated ROI',
    },
    'phe_pseudo_prob_distance_prior_brainwin_skullstrip': {
        'dataset_id': 195,
        'prior_channels': ['ich_probability', 'ich_distance'],
        'ct_input_windows': ['brain'],
        'short_name': 'bwss_probdist',
        'dataset_suffix': 'PHESICH_PHE_19b_bwss',
        'pseudo_ich_output_name': 'ct_prior_bwss',
        'description': 'Brain-window/skull-stripped CT + CT-only pseudo ICH probability + distance prior',
    },
    'phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu': {
        'dataset_id': 196,
        'prior_channels': ['ich_probability', 'ich_distance'],
        'ct_input_windows': ['brain', 'subdural'],
        'short_name': 'bw55otsu',
        'dataset_suffix': 'PHESICH_PHE_19b_bw55',
        'pseudo_ich_output_name': 'ct_prior_bw55otsu',
        'description': 'Brain+subdural window CT + fixed 55 HU/local Otsu pseudo ICH probability + distance prior',
    },
    'phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval': {
        'dataset_id': 198,
        'prior_channels': ['ich_probability', 'ich_distance'],
        'ct_input_windows': ['brain', 'subdural'],
        'short_name': '19c_hardval_bw55otsu',
        'dataset_suffix': 'PHESICH_PHE_19c_hardval_bw55',
        'pseudo_ich_output_name': 'ct_prior_bw55otsu',
        'description': '02_19c diagnostic resplit: brain+subdural CT + fixed 55 HU/local Otsu pseudo ICH probability + distance prior',
    },
    'phe_pseudo_prob_distance_prior_triple_window_ct55_otsu': {
        'dataset_id': 197,
        'prior_channels': ['ich_probability', 'ich_distance'],
        'ct_input_windows': ['brain', 'subdural', 'bone'],
        'short_name': 'tw55otsu',
        'dataset_suffix': 'PHESICH_PHE_19b_tw55',
        'pseudo_ich_output_name': 'ct_prior_tw55otsu',
        'description': 'Brain+subdural+bone window CT + fixed 55 HU/local Otsu pseudo ICH probability + distance prior',
    },
}

EXPERIMENT_ID = 'phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval'
assert EXPERIMENT_ID in EXPERIMENTS, f'Unknown EXPERIMENT_ID: {EXPERIMENT_ID}'
EXPERIMENT = EXPERIMENTS[EXPERIMENT_ID]
PRIOR_CHANNELS = list(EXPERIMENT['prior_channels'])
CT_INPUT_WINDOW_NAMES = list(EXPERIMENT.get('ct_input_windows', ['brain']))
missing_ct_windows = [name for name in CT_INPUT_WINDOW_NAMES if name not in CT_WINDOW_LIBRARY]
assert not missing_ct_windows, f'Unknown CT input windows: {missing_ct_windows}'
CT_INPUT_WINDOWS = []
for window_name in CT_INPUT_WINDOW_NAMES:
    spec = dict(CT_WINDOW_LIBRARY[window_name])
    spec['name'] = window_name
    spec['low'] = float(spec['level']) - float(spec['width']) / 2.0
    spec['high'] = float(spec['level']) + float(spec['width']) / 2.0
    CT_INPUT_WINDOWS.append(spec)

OUTPUT_BASE = WORK_ROOT / 'outputs_02_19c'
EXPERIMENT_SHORT_NAME = EXPERIMENT.get('short_name', EXPERIMENT_ID)
OUTPUT_ROOT = OUTPUT_BASE / EXPERIMENT_SHORT_NAME
PSEUDO_ICH_OUTPUT_NAME = EXPERIMENT.get('pseudo_ich_output_name', 'phe_sich_ct_threshold_prior')
PSEUDO_ICH_OUTPUT_ROOT = OUTPUT_BASE / 'pseudo_ich_outputs' / PSEUDO_ICH_OUTPUT_NAME
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
# Keep results short on Windows to avoid long path failures during checkpointing.
NNUNET_RESULTS = WORK_ROOT / 'o19c_results' / EXPERIMENT_SHORT_NAME

DATASET_ID = int(EXPERIMENT['dataset_id'])
DATASET_NAME = f'Dataset{DATASET_ID:03d}_{EXPERIMENT.get("dataset_suffix", "PHESICH_PHE_" + EXPERIMENT_SHORT_NAME)}'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

ICH_PRIOR_PROB_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'probability_maps'
ICH_PRIOR_MASK_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'binary_masks'
ICH_DISTANCE_PRIOR_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'distance_maps'
ICH_DILATED_ROI_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'dilated_roi'
PRIOR_QC_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'visualizations'


def strip_nii_suffix(path_like):
    name = Path(str(path_like)).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return Path(name).stem


def scan_id_from_value(value):
    text = strip_nii_suffix(value).strip()
    if re.fullmatch(r'\d+\.0+', text):
        text = text.split('.')[0]
    m = re.search(r'(\d{4})$', text)
    if not m:
        m = re.search(r'(\d+)$', text)
    if not m:
        raise ValueError(f'Cannot parse scan id from {value}')
    return m.group(1).zfill(4)


def make_case_id(scan_id):
    scan_id = str(scan_id).strip()
    if scan_id.lower().startswith('phe_'):
        return scan_id
    if scan_id.isdigit():
        return f'phe_{int(scan_id):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in scan_id).strip('_')
    return f'phe_{safe}'


def split_from_scan_id(scan_id):
    scan_id4 = scan_id_from_value(scan_id)
    if scan_id4 in TEST_SCAN_IDS:
        return 'test'
    if scan_id4 in VAL_SCAN_IDS:
        return 'val'
    return 'train'


def has_nifti_pair_root(path_like):
    root = Path(path_like)
    if not (root / 'set').exists() or not (root / 'label').exists():
        return False
    image_count = len(list((root / 'set').glob('*.nii'))) + len(list((root / 'set').glob('*.nii.gz')))
    mask_count = len(list((root / 'label').glob('*.nii'))) + len(list((root / 'label').glob('*.nii.gz')))
    return image_count > 0 and mask_count > 0


def find_nnunet_repo():
    if USER_NNUNET_REPO is not None:
        repo = Path(USER_NNUNET_REPO)
        if (repo / 'nnunetv2' / '__init__.py').exists():
            return repo
        raise FileNotFoundError(f'USER_NNUNET_REPO is not an nnU-Net repo: {repo}')
    candidates = [LOCAL_ROOT / 'nnUNet-master', LOCAL_ROOT]
    if IS_KAGGLE:
        candidates.extend(KAGGLE_INPUT.iterdir())
    for base in candidates:
        if (base / 'nnunetv2' / '__init__.py').exists():
            return base
        if (base / 'nnUNet-master' / 'nnunetv2' / '__init__.py').exists():
            return base / 'nnUNet-master'
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    for root in search_roots:
        for init_file in root.rglob('nnunetv2/__init__.py'):
            return init_file.parents[1]
    raise FileNotFoundError('Could not find local nnU-Net repo. Set USER_NNUNET_REPO to your nnUNet-master folder.')


def find_phe_root():
    if USER_PHE_ROOT is not None:
        root = Path(USER_PHE_ROOT)
        if has_nifti_pair_root(root):
            return root
        raise FileNotFoundError(f'USER_PHE_ROOT does not contain set/label NIfTI pairs: {root}')
    candidates = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        LOCAL_ROOT / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([
                base / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'NIFIT',
                base,
            ])
    for root in candidates:
        if has_nifti_pair_root(root):
            return root
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    found = []
    for root in search_roots:
        for set_dir in root.rglob('set'):
            candidate = set_dir.parent
            if has_nifti_pair_root(candidate):
                found.append(candidate)
    if found:
        found = sorted(found, key=lambda x: (('SubdatasetA_NIFIT' not in str(x)), len(str(x))))
        return found[0]
    raise FileNotFoundError('Could not find PHE NIfTI root with set/ and label/. Set USER_PHE_ROOT manually.')


def find_split_csv():
    if USER_SPLIT_CSV is not None and Path(USER_SPLIT_CSV).exists():
        return Path(USER_SPLIT_CSV)
    names = {'3dff_iph_phe_patient_split.csv', 'phe_split.csv', 'patient_split.csv'}
    candidates = [
        LOCAL_ROOT / 'outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation' / 'manifests' / '3dff_iph_phe_patient_split.csv'
    ]
    if IS_KAGGLE:
        for root in KAGGLE_INPUT.iterdir():
            for csv_path in root.rglob('*.csv'):
                if csv_path.name in names:
                    candidates.append(csv_path)
    for csv_path in candidates:
        if csv_path.exists():
            return csv_path
    return None


def build_nifti_index(folder):
    files = sorted(list(Path(folder).glob('*.nii')) + list(Path(folder).glob('*.nii.gz')))
    out = {}
    for file_path in files:
        try:
            out[scan_id_from_value(file_path)] = file_path
        except ValueError:
            pass
    return out


def find_nii_by_scan(folder, scan_id):
    scan_id4 = scan_id_from_value(scan_id)
    direct_candidates = [
        Path(folder) / f'{scan_id4}.nii.gz',
        Path(folder) / f'{scan_id4}.nii',
        Path(folder) / f'{int(scan_id4)}.nii.gz',
        Path(folder) / f'{int(scan_id4)}.nii',
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate
    indexed = build_nifti_index(folder)
    if scan_id4 in indexed:
        return indexed[scan_id4]
    raise FileNotFoundError(f'Missing NIfTI for scan_id={scan_id4} in {folder}')


NNUNET_REPO = find_nnunet_repo()
PHE_ROOT = find_phe_root()
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'
SPLIT_CSV = find_split_csv()

for path in [
    OUTPUT_BASE, OUTPUT_ROOT, PSEUDO_ICH_OUTPUT_ROOT,
    NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS,
    ICH_PRIOR_PROB_DIR, ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR,
    ICH_DILATED_ROI_DIR, PRIOR_QC_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

if str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

print('IS_KAGGLE            :', IS_KAGGLE)
print('WORK_ROOT            :', WORK_ROOT)
print('Experiment           :', EXPERIMENT_ID, '-', EXPERIMENT['description'])
print('Short output name    :', EXPERIMENT_SHORT_NAME)
print('Prior channels       :', PRIOR_CHANNELS)
print('CT input windows     :', [(w['name'], w['level'], w['width']) for w in CT_INPUT_WINDOWS])
print('Pseudo ICH source    :', PSEUDO_ICH_SOURCE)
print('CT preprocess profile:', CT_PREPROCESS_PROFILE)
print('CT brain window HU   :', (CT_WINDOW_LOW_HU, CT_WINDOW_HIGH_HU))
print('CT skull stripping   :', CT_SKULL_STRIP_ENABLED)
print('nnU-Net repo         :', NNUNET_REPO)
print('PHE root             :', PHE_ROOT)
print('PHE image dir        :', PHE_IMAGE_DIR)
print('PHE mask dir         :', PHE_MASK_DIR)
print('Split CSV            :', SPLIT_CSV if SPLIT_CSV is not None else 'not found, using embedded split IDs')
print('19c validation IDs   :', sorted(VAL_SCAN_IDS))
print('19c test IDs         :', sorted(TEST_SCAN_IDS))
print('Images               :', len(list(PHE_IMAGE_DIR.glob('*.nii*'))))
print('Masks                :', len(list(PHE_MASK_DIR.glob('*.nii*'))))
print('Output root          :', OUTPUT_ROOT)
print('Pseudo ICH output    :', PSEUDO_ICH_OUTPUT_ROOT)
print('Dataset              :', DATASET_NAME)
print('nnUNet_raw           :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed  :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results       :', os.environ['nnUNet_results'])


IS_KAGGLE            : False
WORK_ROOT            : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao
Experiment           : phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval - 02_19c diagnostic resplit: brain+subdural CT + fixed 55 HU/local Otsu pseudo ICH probability + distance prior
Short output name    : 19c_hardval_bw55otsu
Prior channels       : ['ich_probability', 'ich_distance']
CT input windows     : [('brain', 40.0, 120.0), ('subdural', 75.0, 215.0)]
Pseudo ICH source    : ct_fixed55_localotsu_no_gt
CT preprocess profile: brain_subdural_window_l40w120_ct55_otsu_skullstrip_v2
CT brain window HU   : (-20.0, 100.0)
CT skull stripping   : True
nnU-Net repo         : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master
PHE root             : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-IDS\SubdatasetA_NIFIT\NIFIT
PHE image dir        : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-IDS\SubdatasetA_NIFIT\NIFIT\set
PHE mask dir         : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-C

## 0. Local path setup

This notebook expects local workspace folders:

- `nnUNet-master/` containing `nnunetv2/__init__.py`
- `PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT/set` and `label`

If `USER_SPLIT_CSV` is `None`, the notebook uses the embedded fair val/test IDs used by the recent PHE-SICH nnU-Net baselines.


In [2]:
AUTO_INSTALL_MISSING_DEPS = False
INSTALL_NNUNET_EDITABLE = False


def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError as exc:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise ModuleNotFoundError(
                f'Missing {import_name}. Use the Python 3.11 kernel/environment that already runs nnU-Net, '
                f'or set AUTO_INSTALL_MISSING_DEPS=True in this cell to install {pip_name}.'
            ) from exc
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)


if INSTALL_NNUNET_EDITABLE:
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

ensure_import('SimpleITK')
ensure_import('scipy')
ensure_import('batchgenerators')
ensure_import('batchgeneratorsv2')
ensure_import('acvl_utils', 'acvl-utils')
ensure_import('dynamic_network_architectures', 'dynamic-network-architectures')
ensure_import('blosc2')
ensure_import('yacs')
ensure_import('nnunetv2')

import SimpleITK as sitk
from scipy import ndimage as ndi
import nnunetv2
print('SimpleITK OK')
print('scipy OK')
print('nnunetv2 import OK:', nnunetv2.__file__)


SimpleITK OK
scipy OK
nnunetv2 import OK: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master\nnunetv2\__init__.py


## 1. Build PHE-SICH split manifest

If `USER_SPLIT_CSV` is present, its split labels are reused but image/mask paths are remapped to the local PHE-SICH dataset. If no CSV is present, the notebook uses the same embedded split IDs as the recent PHE-SICH nnU-Net baselines: train 99, val 10, test 11.


In [3]:
image_by_scan = build_nifti_index(PHE_IMAGE_DIR)
mask_by_scan = build_nifti_index(PHE_MASK_DIR)
assert image_by_scan, f'No NIfTI images found in {PHE_IMAGE_DIR}'
assert mask_by_scan, f'No NIfTI masks found in {PHE_MASK_DIR}'

rows = []
if SPLIT_CSV is not None:
    raw_split_df = pd.read_csv(SPLIT_CSV, dtype={'scan_id': str, 'patient_id': str})
    raw_split_df.columns = [str(c).strip() for c in raw_split_df.columns]
    for raw_row in raw_split_df.to_dict('records'):
        scan_source = raw_row.get('scan_id', None)
        if scan_source is None or pd.isna(scan_source) or str(scan_source).strip() == '':
            scan_source = raw_row.get('patient_id', None)
        if scan_source is None or pd.isna(scan_source) or str(scan_source).strip() == '':
            scan_source = raw_row.get('img_path', raw_row.get('image_path', ''))
        scan_id = scan_id_from_value(scan_source)
        split_value = str(raw_row.get('split', split_from_scan_id(scan_id))).strip().lower()
        if split_value not in {'train', 'val', 'test'}:
            split_value = split_from_scan_id(scan_id)

        img_path = Path(str(raw_row.get('img_path', raw_row.get('image_path', ''))))
        mask_path = Path(str(raw_row.get('phe_mask_path', raw_row.get('label_path', raw_row.get('mask_path', '')))))
        if not img_path.exists():
            img_path = image_by_scan.get(scan_id) or find_nii_by_scan(PHE_IMAGE_DIR, scan_id)
        if not mask_path.exists():
            mask_path = mask_by_scan.get(scan_id) or find_nii_by_scan(PHE_MASK_DIR, scan_id)
        rows.append({
            'scan_id': scan_id,
            'case_id': make_case_id(scan_id),
            'split': split_value,
            'img_path': str(img_path),
            'phe_mask_path': str(mask_path),
        })
else:
    for scan_id, img_path in sorted(image_by_scan.items()):
        mask_path = mask_by_scan.get(scan_id)
        if mask_path is None:
            print('WARNING: missing mask for image scan_id:', scan_id, img_path)
            continue
        rows.append({
            'scan_id': scan_id,
            'case_id': make_case_id(scan_id),
            'split': split_from_scan_id(scan_id),
            'img_path': str(img_path),
            'phe_mask_path': str(mask_path),
        })

split_df = pd.DataFrame(rows).drop_duplicates('case_id', keep='first').sort_values(['split', 'scan_id']).reset_index(drop=True)
required_cols = {'case_id', 'split', 'img_path', 'phe_mask_path'}
missing_cols = required_cols - set(split_df.columns)
assert not missing_cols, f'Missing columns in split manifest: {missing_cols}'
assert split_df['case_id'].is_unique, 'case_id must be unique.'
assert split_df['img_path'].map(lambda x: Path(x).exists()).all(), 'Some image paths are missing.'
assert split_df['phe_mask_path'].map(lambda x: Path(x).exists()).all(), 'Some PHE mask paths are missing.'

manifest_csv = OUTPUT_ROOT / '02_19c_local_phe_sich_split_manifest.csv'
split_df.to_csv(manifest_csv, index=False)

print('Total cases:', len(split_df))
print('Manifest:', manifest_csv)
display(split_df.groupby('split').size().rename('cases').reset_index())
display(split_df.head())


Total cases: 120
Manifest: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_local_phe_sich_split_manifest.csv


,split,cases
0,test,11
1,train,99
2,val,10


,scan_id,case_id,split,img_path,phe_mask_path
0,0002,phe_0002,test,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
1,0029,phe_0029,test,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
2,0033,phe_0033,test,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
3,0080,phe_0080,test,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
4,0084,phe_0084,test,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...


## 2. Generate pseudo-ICH priors directly on PHE-SICH CT

This control replaces the `02_13b` teacher with a deterministic CT-only pseudo-labeler. It intentionally avoids true ICH labels and avoids PHE masks when creating prior channels, so the comparison with `02_14b` stays fair.

The v2 prior has four explicit CT-only branches: fixed global 55 HU, local Otsu inside a CT-derived high-density ROI, >80 HU artifact control by adjacency to the 40-80 HU hematoma core, and standardized morphology/component filtering.


In [4]:
RUN_PSEUDO_ICH_PRIOR_GENERATION = True
OVERWRITE_PSEUDO_ICH_PRIORS = False

# CT-only pseudo-ICH policy. Keep these fixed when comparing 19b against 14b/19.
PSEUDO_ICH_SOFT_LOW_HU = 35.0
PSEUDO_ICH_CORE_LOW_HU = 40.0
PSEUDO_ICH_FIXED_THRESHOLD_HU = 55.0
PSEUDO_ICH_CORE_HIGH_HU = 80.0
PSEUDO_ICH_SOFT_HIGH_HU = 100.0
PSEUDO_ICH_HIGH_HU_ALLOWED_MAX = 110.0
PSEUDO_ICH_MIN_VOLUME_ML = 0.20
PSEUDO_ICH_MAX_VOLUME_ML = 120.0
PSEUDO_ICH_KEEP_LARGEST_COMPONENTS = 3

# Local adaptive threshold branch. ROI comes from CT only, not from PHE masks.
PSEUDO_ICH_LOCAL_OTSU_ENABLED = True
PSEUDO_ICH_LOCAL_OTSU_DILATE_RADIUS_MM = 8.0
PSEUDO_ICH_LOCAL_OTSU_MIN_VOXELS = 64
PSEUDO_ICH_LOCAL_OTSU_NBINS = 128
PSEUDO_ICH_LOCAL_OTSU_CLIP_LOW_HU = 40.0
PSEUDO_ICH_LOCAL_OTSU_CLIP_HIGH_HU = 80.0

# >80 HU values are treated as possible calcification/artifact unless adjacent to the 40-80 HU core.
PSEUDO_ICH_HIGH_HU_ARTIFACT_CUTOFF = 80.0
PSEUDO_ICH_HIGH_HU_ADJ_RADIUS_MM = 3.0

# Paper-style morphology: closing 5x5, then opening/removing small islands with 7x7.
PSEUDO_ICH_MORPH_CLOSING_KERNEL_PX = 5
PSEUDO_ICH_MORPH_OPENING_KERNEL_PX = 7
PSEUDO_ICH_SOFT_SUPPORT_RADIUS_MM = 2.0
DISTANCE_SCALE_MM = 10.0
DILATED_ROI_RADIUS_MM = 20.0

# Use an intracranial mask to suppress skull/scalp/calcification-like false pseudo-ICH candidates.
PSEUDO_ICH_USE_BRAIN_MASK = True

for p in [ICH_PRIOR_PROB_DIR, ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR, ICH_DILATED_ROI_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if OVERWRITE_PSEUDO_ICH_PRIORS:
    for p in [ICH_PRIOR_PROB_DIR, ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR, ICH_DILATED_ROI_DIR]:
        for f in p.glob('*.nii.gz'):
            f.unlink()


def write_array_like_reference(arr, reference_path: Path, out_path: Path, dtype=np.float32):
    ref = sitk.ReadImage(str(reference_path))
    out = sitk.GetImageFromArray(np.asarray(arr).astype(dtype))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))


def make_ball_structure(spacing_xyz, radius_mm):
    radius_mm = float(radius_mm)
    if radius_mm <= 0:
        return np.ones((1, 1, 1), dtype=bool)
    spacing_zyx = np.asarray(spacing_xyz[::-1], dtype=float)
    radii = np.maximum(1, np.ceil(radius_mm / np.maximum(spacing_zyx, 1e-6)).astype(int))
    zz, yy, xx = np.ogrid[-radii[0]:radii[0] + 1, -radii[1]:radii[1] + 1, -radii[2]:radii[2] + 1]
    dist_mm = np.sqrt((zz * spacing_zyx[0]) ** 2 + (yy * spacing_zyx[1]) ** 2 + (xx * spacing_zyx[2]) ** 2)
    return dist_mm <= radius_mm


def make_plane_square_structure(kernel_px):
    kernel_px = int(kernel_px)
    if kernel_px <= 1:
        return np.ones((1, 1, 1), dtype=bool)
    if kernel_px % 2 == 0:
        kernel_px += 1
    return np.ones((1, kernel_px, kernel_px), dtype=bool)


def _largest_or_center_component_2d(mask2d, min_voxels=500):
    mask2d = np.asarray(mask2d, dtype=bool)
    labeled, num = ndi.label(mask2d)
    if num == 0:
        return np.zeros(mask2d.shape, dtype=bool)
    yy, xx = np.indices(mask2d.shape)
    center = np.asarray(mask2d.shape, dtype=float) / 2.0
    best_label = None
    best_score = -np.inf
    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        voxels = int(comp.sum())
        if voxels < int(min_voxels):
            continue
        cy = float(yy[comp].mean())
        cx = float(xx[comp].mean())
        dist = np.sqrt((cy - center[0]) ** 2 + (cx - center[1]) ** 2)
        score = voxels / (1.0 + 0.02 * dist)
        if score > best_score:
            best_score = score
            best_label = comp_id
    if best_label is None:
        counts = np.bincount(labeled.ravel())
        counts[0] = 0
        best_label = int(counts.argmax())
    return labeled == best_label


def make_intracranial_mask(ct_arr):
    ct = np.nan_to_num(np.asarray(ct_arr, dtype=np.float32), nan=-1024.0, posinf=3071.0, neginf=-1024.0)
    if not CT_SKULL_STRIP_ENABLED:
        return np.ones(ct.shape, dtype=bool)
    out = np.zeros(ct.shape, dtype=bool)
    for z in range(ct.shape[0]):
        sl = ct[z]
        head = sl > float(CT_SKULL_STRIP_HEAD_LOW_HU)
        if not head.any():
            continue
        head = ndi.binary_fill_holes(ndi.binary_closing(head, iterations=2))
        head = _largest_or_center_component_2d(head, min_voxels=CT_SKULL_STRIP_MIN_COMPONENT_VOXELS)
        bone = sl > float(CT_SKULL_STRIP_BONE_HU)
        if CT_SKULL_STRIP_BONE_DILATE_ITER > 0:
            bone = ndi.binary_dilation(bone, iterations=int(CT_SKULL_STRIP_BONE_DILATE_ITER))
        brain_candidate = head & ~bone
        brain_candidate = ndi.binary_opening(brain_candidate, iterations=1)
        brain_candidate = _largest_or_center_component_2d(brain_candidate, min_voxels=CT_SKULL_STRIP_MIN_COMPONENT_VOXELS)
        if brain_candidate.any():
            brain_candidate = ndi.binary_fill_holes(brain_candidate)
        out[z] = brain_candidate
    return out.astype(bool)


def window_ct_to_scale(ct_arr, level_hu, width_hu, output_scale=100.0):
    low = float(level_hu) - float(width_hu) / 2.0
    high = float(level_hu) + float(width_hu) / 2.0
    ct = np.nan_to_num(np.asarray(ct_arr, dtype=np.float32), nan=low, posinf=high, neginf=low)
    clipped = np.clip(ct, low, high)
    scaled = (clipped - low) / max(high - low, 1e-6)
    return (scaled * float(output_scale)).astype(np.float32), low, high


def preprocess_ct_window_for_phe_input(ct_arr, window_spec, brain_mask=None):
    if brain_mask is None:
        brain_mask = make_intracranial_mask(ct_arr)
    scaled, low, high = window_ct_to_scale(
        ct_arr,
        window_spec['level'],
        window_spec['width'],
        output_scale=CT_WINDOW_OUTPUT_SCALE,
    )
    if CT_SKULL_STRIP_ENABLED:
        scaled = np.where(brain_mask, scaled, 0.0).astype(np.float32)
    name = str(window_spec['name'])
    stats = {
        f'ct_{name}_window_low_hu': float(low),
        f'ct_{name}_window_high_hu': float(high),
        f'ct_{name}_channel_min': float(scaled.min()),
        f'ct_{name}_channel_max': float(scaled.max()),
        f'ct_{name}_channel_mean': float(scaled.mean()),
    }
    return scaled, brain_mask, stats


def preprocess_ct_for_phe_input(ct_arr):
    brain_mask = make_intracranial_mask(ct_arr)
    brain_spec = dict(CT_WINDOW_LIBRARY['brain'])
    brain_spec['name'] = 'brain'
    scaled, _, stats = preprocess_ct_window_for_phe_input(ct_arr, brain_spec, brain_mask=brain_mask)
    stats.update({
        'ct_preprocess_profile': CT_PREPROCESS_PROFILE,
        'ct_window_low_hu': float(CT_WINDOW_LOW_HU),
        'ct_window_high_hu': float(CT_WINDOW_HIGH_HU),
        'brain_mask_voxels': int(brain_mask.sum()),
        'ct_channel_min': float(scaled.min()),
        'ct_channel_max': float(scaled.max()),
        'ct_channel_mean': float(scaled.mean()),
    })
    return scaled, brain_mask, stats


def trapezoid_probability(ct_arr, soft_low, core_low, core_high, soft_high):
    ct = np.asarray(ct_arr, dtype=np.float32)
    prob = np.zeros(ct.shape, dtype=np.float32)
    rising = (ct >= soft_low) & (ct < core_low)
    core = (ct >= core_low) & (ct <= core_high)
    falling = (ct > core_high) & (ct <= soft_high)
    prob[rising] = (ct[rising] - soft_low) / max(core_low - soft_low, 1e-6)
    prob[core] = 1.0
    prob[falling] = (soft_high - ct[falling]) / max(soft_high - core_high, 1e-6)
    return np.clip(prob, 0.0, 1.0).astype(np.float32)


def otsu_threshold_numpy(values, nbins=128, clip_low=None, clip_high=None):
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]
    if clip_low is not None:
        values = values[values >= float(clip_low)]
    if clip_high is not None:
        values = values[values <= float(clip_high)]
    if values.size < 2:
        return None
    vmin = float(values.min()) if clip_low is None else float(clip_low)
    vmax = float(values.max()) if clip_high is None else float(clip_high)
    if vmax <= vmin:
        return None
    hist, edges = np.histogram(values, bins=int(nbins), range=(vmin, vmax))
    hist = hist.astype(np.float64)
    total = hist.sum()
    if total <= 0:
        return None
    centers = (edges[:-1] + edges[1:]) / 2.0
    weight1 = np.cumsum(hist)
    weight2 = total - weight1
    mean1 = np.cumsum(hist * centers) / np.maximum(weight1, 1e-12)
    mean2 = (np.cumsum((hist * centers)[::-1]) / np.maximum(np.cumsum(hist[::-1]), 1e-12))[::-1]
    variance12 = weight1[:-1] * weight2[:-1] * (mean1[:-1] - mean2[1:]) ** 2
    if variance12.size == 0 or not np.isfinite(variance12).any():
        return None
    idx = int(np.nanargmax(variance12))
    return float(centers[idx])


def filter_pseudo_ich_components(candidate, spacing_xyz, min_volume_ml, max_volume_ml, keep_largest):
    candidate = np.asarray(candidate, dtype=bool)
    if not candidate.any():
        return np.zeros(candidate.shape, dtype=np.uint8), []
    labeled, num = ndi.label(candidate)
    voxel_ml = float(np.prod(np.asarray(spacing_xyz, dtype=float)) / 1000.0)
    components = []
    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        voxels = int(comp.sum())
        volume_ml = voxels * voxel_ml
        if volume_ml < float(min_volume_ml) or volume_ml > float(max_volume_ml):
            continue
        components.append((comp_id, voxels, volume_ml))
    components = sorted(components, key=lambda x: x[1], reverse=True)[:int(keep_largest)]
    out = np.zeros(candidate.shape, dtype=bool)
    for comp_id, _, _ in components:
        out |= labeled == comp_id
    return out.astype(np.uint8), [
        {'component_id': int(comp_id), 'voxels': int(voxels), 'volume_ml': float(volume_ml)}
        for comp_id, voxels, volume_ml in components
    ]


def make_distance_prior(binary_ich, spacing_xyz, scale_mm=10.0):
    binary_ich = np.asarray(binary_ich).astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.float32)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return np.exp(-dist_mm / float(scale_mm)).astype(np.float32)


def make_dilated_roi(binary_ich, spacing_xyz, radius_mm=20.0):
    binary_ich = np.asarray(binary_ich).astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.uint8)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return (dist_mm <= float(radius_mm)).astype(np.uint8)


def apply_pseudo_ich_morphology(candidate, brain_mask):
    candidate = np.asarray(candidate, dtype=bool) & np.asarray(brain_mask, dtype=bool)
    if not candidate.any():
        return candidate, {'morph_fallback_opening_empty': False}
    close_structure = make_plane_square_structure(PSEUDO_ICH_MORPH_CLOSING_KERNEL_PX)
    open_structure = make_plane_square_structure(PSEUDO_ICH_MORPH_OPENING_KERNEL_PX)
    closed = ndi.binary_closing(candidate, structure=close_structure)
    filled = np.zeros_like(closed, dtype=bool)
    for z in range(closed.shape[0]):
        filled[z] = ndi.binary_fill_holes(closed[z])
    opened = ndi.binary_opening(filled, structure=open_structure)
    fallback = bool(filled.any() and not opened.any())
    morphed = filled if fallback else opened
    morphed &= brain_mask
    return morphed.astype(bool), {'morph_fallback_opening_empty': fallback}


def generate_pseudo_ich_from_ct(ct_arr, spacing_xyz):
    ct = np.nan_to_num(np.asarray(ct_arr, dtype=np.float32), nan=-1024.0, posinf=3071.0, neginf=-1024.0)
    brain_mask = make_intracranial_mask(ct) if PSEUDO_ICH_USE_BRAIN_MASK else np.ones(ct.shape, dtype=bool)

    mid_density = (ct >= PSEUDO_ICH_CORE_LOW_HU) & (ct <= PSEUDO_ICH_CORE_HIGH_HU) & brain_mask
    very_high = (ct > PSEUDO_ICH_HIGH_HU_ARTIFACT_CUTOFF) & (ct <= PSEUDO_ICH_HIGH_HU_ALLOWED_MAX) & brain_mask
    adjacency_structure = make_ball_structure(spacing_xyz, PSEUDO_ICH_HIGH_HU_ADJ_RADIUS_MM)
    mid_density_large = ndi.binary_opening(mid_density, structure=make_plane_square_structure(3)) if mid_density.any() else mid_density
    if not mid_density_large.any():
        mid_density_large = mid_density
    very_high_kept = very_high & ndi.binary_dilation(mid_density_large, structure=adjacency_structure)
    density_eligible = mid_density | very_high_kept

    fixed55_candidate = (ct >= PSEUDO_ICH_FIXED_THRESHOLD_HU) & density_eligible & brain_mask

    local_otsu_threshold = None
    local_otsu_candidate = np.zeros(ct.shape, dtype=bool)
    if PSEUDO_ICH_LOCAL_OTSU_ENABLED and fixed55_candidate.any():
        otsu_roi_structure = make_ball_structure(spacing_xyz, PSEUDO_ICH_LOCAL_OTSU_DILATE_RADIUS_MM)
        otsu_roi = ndi.binary_dilation(fixed55_candidate, structure=otsu_roi_structure)
        otsu_roi &= brain_mask
        otsu_roi &= ct >= PSEUDO_ICH_SOFT_LOW_HU
        otsu_roi &= ct <= PSEUDO_ICH_SOFT_HIGH_HU
        otsu_values = ct[otsu_roi]
        if int(otsu_values.size) >= int(PSEUDO_ICH_LOCAL_OTSU_MIN_VOXELS):
            local_otsu_threshold = otsu_threshold_numpy(
                otsu_values,
                nbins=PSEUDO_ICH_LOCAL_OTSU_NBINS,
                clip_low=PSEUDO_ICH_LOCAL_OTSU_CLIP_LOW_HU,
                clip_high=PSEUDO_ICH_LOCAL_OTSU_CLIP_HIGH_HU,
            )
        if local_otsu_threshold is not None:
            local_otsu_threshold = float(np.clip(
                local_otsu_threshold,
                PSEUDO_ICH_LOCAL_OTSU_CLIP_LOW_HU,
                PSEUDO_ICH_LOCAL_OTSU_CLIP_HIGH_HU,
            ))
            local_otsu_candidate = (ct >= local_otsu_threshold) & density_eligible & otsu_roi & brain_mask

    candidate_pre_morph = (fixed55_candidate | local_otsu_candidate) & brain_mask
    candidate_morph, morph_diag = apply_pseudo_ich_morphology(candidate_pre_morph, brain_mask)
    binary, components = filter_pseudo_ich_components(
        candidate_morph,
        spacing_xyz,
        min_volume_ml=PSEUDO_ICH_MIN_VOLUME_ML,
        max_volume_ml=PSEUDO_ICH_MAX_VOLUME_ML,
        keep_largest=PSEUDO_ICH_KEEP_LARGEST_COMPONENTS,
    )

    intensity_prob = trapezoid_probability(
        ct,
        PSEUDO_ICH_SOFT_LOW_HU,
        PSEUDO_ICH_CORE_LOW_HU,
        PSEUDO_ICH_CORE_HIGH_HU,
        PSEUDO_ICH_SOFT_HIGH_HU,
    )
    intensity_prob = np.where(density_eligible & brain_mask, intensity_prob, 0.0).astype(np.float32)
    support = make_dilated_roi(binary, spacing_xyz, radius_mm=PSEUDO_ICH_SOFT_SUPPORT_RADIUS_MM).astype(bool)
    support &= brain_mask
    probability = np.where(support, intensity_prob, 0.0).astype(np.float32)
    probability[fixed55_candidate] = np.maximum(probability[fixed55_candidate], 0.65)
    probability[local_otsu_candidate] = np.maximum(probability[local_otsu_candidate], 0.60)
    probability[binary.astype(bool)] = np.maximum(probability[binary.astype(bool)], 0.85)
    probability[very_high & ~very_high_kept] = 0.0
    probability = np.clip(probability, 0.0, 1.0).astype(np.float32)

    diagnostics = {
        'fixed55_voxels': int(fixed55_candidate.sum()),
        'local_otsu_threshold_hu': float(local_otsu_threshold) if local_otsu_threshold is not None else np.nan,
        'local_otsu_voxels': int(local_otsu_candidate.sum()),
        'mid_density_voxels': int(mid_density.sum()),
        'very_high_voxels': int(very_high.sum()),
        'very_high_kept_voxels': int(very_high_kept.sum()),
        'very_high_removed_voxels': int((very_high & ~very_high_kept).sum()),
        'candidate_pre_morph_voxels': int(candidate_pre_morph.sum()),
        'candidate_post_morph_voxels': int(candidate_morph.sum()),
        'morph_closing_kernel_px': int(PSEUDO_ICH_MORPH_CLOSING_KERNEL_PX),
        'morph_opening_kernel_px': int(PSEUDO_ICH_MORPH_OPENING_KERNEL_PX),
        **morph_diag,
    }
    return probability, binary.astype(np.uint8), components, brain_mask, diagnostics


def prior_file_ready(path: Path):
    return path.exists() and path.stat().st_size > 0

expected_prior_groups = []
missing_prior_paths = []
for case_id in split_df['case_id'].astype(str):
    paths = [
        ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz',
        ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz',
        ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz',
        ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz',
    ]
    expected_prior_groups.append(paths)
    missing_prior_paths.extend([p for p in paths if not prior_file_ready(p)])
complete_prior_cases = sum(all(prior_file_ready(p) for p in paths) for paths in expected_prior_groups)
print('Complete pseudo-ICH prior cases:', complete_prior_cases, '/', len(expected_prior_groups))
if missing_prior_paths:
    print('Missing/corrupt prior files:', len(missing_prior_paths))
    print('First missing/corrupt:', missing_prior_paths[0])

prior_rows = []
if RUN_PSEUDO_ICH_PRIOR_GENERATION and (OVERWRITE_PSEUDO_ICH_PRIORS or missing_prior_paths):
    for row in split_df.itertuples(index=False):
        case_id = str(row.case_id)
        img_path = Path(row.img_path)
        img = sitk.ReadImage(str(img_path))
        ct = sitk.GetArrayFromImage(img)
        spacing_xyz = np.array(img.GetSpacing(), dtype=float)

        prob, binary, components, brain_mask, diagnostics = generate_pseudo_ich_from_ct(ct, spacing_xyz)
        distance_prior = make_distance_prior(binary, spacing_xyz, scale_mm=DISTANCE_SCALE_MM)
        dilated_roi = make_dilated_roi(binary, spacing_xyz, radius_mm=DILATED_ROI_RADIUS_MM)

        prob_path = ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz'
        binary_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
        distance_path = ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz'
        dilated_path = ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz'
        write_array_like_reference(prob, img_path, prob_path, dtype=np.float32)
        write_array_like_reference(binary, img_path, binary_path, dtype=np.uint8)
        write_array_like_reference(distance_prior, img_path, distance_path, dtype=np.float32)
        write_array_like_reference(dilated_roi, img_path, dilated_path, dtype=np.uint8)

        prior_rows.append({
            'case_id': case_id,
            'split': row.split,
            'source': PSEUDO_ICH_SOURCE,
            'generation_policy': PSEUDO_ICH_POLICY,
            'probability_path': str(prob_path),
            'binary_path': str(binary_path),
            'distance_path': str(distance_path),
            'dilated_roi_path': str(dilated_path),
            'prob_min': float(prob.min()),
            'prob_max': float(prob.max()),
            'prob_mean': float(prob.mean()),
            'binary_voxels': int(binary.sum()),
            'dilated_voxels': int(dilated_roi.sum()),
            'component_count_kept': int(len(components)),
            'component_volumes_ml': '|'.join(f'{c["volume_ml"]:.4f}' for c in components),
            'brain_mask_voxels': int(brain_mask.sum()),
            'skull_strip_enabled': bool(CT_SKULL_STRIP_ENABLED),
            **diagnostics,
        })
elif RUN_PSEUDO_ICH_PRIOR_GENERATION:
    print('All pseudo-ICH priors already exist. Set OVERWRITE_PSEUDO_ICH_PRIORS=True to regenerate.')
else:
    print('Skipped pseudo-ICH prior generation. Set RUN_PSEUDO_ICH_PRIOR_GENERATION=True to run.')

# Always rebuild the manifest from files on disk, so rerunning downstream cells is stable.
manifest_rows = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    prob_path = ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz'
    binary_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
    distance_path = ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz'
    dilated_path = ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz'
    for required_path in [prob_path, binary_path, distance_path, dilated_path]:
        if not prior_file_ready(required_path):
            raise FileNotFoundError(f'Missing or corrupt pseudo-ICH prior for {case_id}: {required_path}')
    prob = sitk.GetArrayFromImage(sitk.ReadImage(str(prob_path)))
    binary = sitk.GetArrayFromImage(sitk.ReadImage(str(binary_path))) > 0
    dilated_roi = sitk.GetArrayFromImage(sitk.ReadImage(str(dilated_path))) > 0

    diag = {}
    try:
        img = sitk.ReadImage(str(row.img_path))
        ct = sitk.GetArrayFromImage(img)
        spacing_xyz = np.array(img.GetSpacing(), dtype=float)
        _, _, components, brain_mask, diag = generate_pseudo_ich_from_ct(ct, spacing_xyz)
        diag['component_count_kept'] = int(len(components))
        diag['component_volumes_ml'] = '|'.join(f'{c["volume_ml"]:.4f}' for c in components)
        diag['brain_mask_voxels'] = int(brain_mask.sum())
    except Exception as exc:
        diag = {'diagnostic_error': repr(exc)}

    manifest_rows.append({
        'case_id': case_id,
        'split': row.split,
        'source': PSEUDO_ICH_SOURCE,
        'generation_policy': PSEUDO_ICH_POLICY,
        'probability_path': str(prob_path),
        'binary_path': str(binary_path),
        'distance_path': str(distance_path),
        'dilated_roi_path': str(dilated_path),
        'prob_min': float(prob.min()),
        'prob_max': float(prob.max()),
        'prob_mean': float(prob.mean()),
        'binary_voxels': int(binary.sum()),
        'dilated_voxels': int(dilated_roi.sum()),
        **diag,
    })

prior_df = pd.DataFrame(manifest_rows)
prior_manifest = PSEUDO_ICH_OUTPUT_ROOT / '02_19c_pseudo_ich_prior_manifest.csv'
prior_df.to_csv(prior_manifest, index=False)
display(prior_df.groupby('split').agg(
    cases=('case_id', 'count'),
    mean_prob=('prob_mean', 'mean'),
    binary_voxels=('binary_voxels', 'sum'),
    empty_binary_cases=('binary_voxels', lambda s: int((pd.to_numeric(s, errors='coerce') == 0).sum())),
    fixed55_voxels=('fixed55_voxels', 'sum'),
    local_otsu_voxels=('local_otsu_voxels', 'sum'),
    very_high_removed_voxels=('very_high_removed_voxels', 'sum'),
    dilated_voxels=('dilated_voxels', 'sum'),
).reset_index())
print('Saved pseudo-ICH prior manifest:', prior_manifest)
print('Pseudo policy         :', PSEUDO_ICH_POLICY)
print('Probability prior dir:', ICH_PRIOR_PROB_DIR)
print('Binary prior dir     :', ICH_PRIOR_MASK_DIR)
print('Distance prior dir   :', ICH_DISTANCE_PRIOR_DIR)
print('Dilated ROI dir      :', ICH_DILATED_ROI_DIR)


Complete pseudo-ICH prior cases: 0 / 120
Missing/corrupt prior files: 480
First missing/corrupt: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\pseudo_ich_outputs\ct_prior_bw55otsu\probability_maps\phe_0002.nii.gz


,split,cases,mean_prob,binary_voxels,empty_binary_cases,fixed55_voxels,local_otsu_voxels,very_high_removed_voxels,dilated_voxels
0,test,11,0.011285,462694,0,974268,814357,121616,5608810
1,train,99,0.012022,4331435,0,10007686,8812349,1028903,56952668
2,val,10,0.008517,270164,0,688148,590953,91650,4310342


Saved pseudo-ICH prior manifest: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\pseudo_ich_outputs\ct_prior_bw55otsu\02_19c_pseudo_ich_prior_manifest.csv
Pseudo policy         : Pseudo-ICH prior is generated from PHE-SICH CT intensities only: fixed 55 HU global branch, local Otsu inside a CT-derived high-density ROI, >80 HU artifact control by adjacency to 40-80 HU core, and morphology/connected-component filtering. No true ICH labels and no PHE masks are used for prior generation.
Probability prior dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\pseudo_ich_outputs\ct_prior_bw55otsu\probability_maps
Binary prior dir     : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\pseudo_ich_outputs\ct_prior_bw55otsu\binary_masks
Distance prior dir   : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\pseudo_ich_outputs\ct_prior_bw55otsu\distance_maps
Dilated ROI dir      : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\pseudo_ich_outputs\ct_prior_bw55otsu\dilated_roi


## 3. Quick inspect pseudo-ICH prior maps


In [5]:
INSPECT_PRIORS = True
MAX_INSPECT = 8

if INSPECT_PRIORS:
    inspect_rows = []
    for row in split_df.head(MAX_INSPECT).itertuples(index=False):
        case_id = str(row.case_id)
        paths = {
            'probability': ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz',
            'binary': ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz',
            'distance': ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz',
            'dilated_roi': ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz',
        }
        record = {'case_id': case_id, 'split': row.split}
        for name, path in paths.items():
            if path.exists():
                arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
                record[f'{name}_shape_zyx'] = tuple(arr.shape)
                record[f'{name}_min'] = float(np.min(arr))
                record[f'{name}_max'] = float(np.max(arr))
                record[f'{name}_voxels_gt0'] = int((arr > 0).sum())
            else:
                record[f'{name}_status'] = 'missing'
        inspect_rows.append(record)
    display(pd.DataFrame(inspect_rows))
else:
    print('Skipped prior inspection.')


,case_id,split,probability_shape_zyx,probability_min,probability_max,probability_voxels_gt0,binary_shape_zyx,binary_min,binary_max,binary_voxels_gt0,distance_shape_zyx,distance_min,distance_max,distance_voxels_gt0,dilated_roi_shape_zyx,dilated_roi_min,dilated_roi_max,dilated_roi_voxels_gt0
0,phe_0002,test,"(32, 512, 512)",0.0,1.0,71354,"(32, 512, 512)",0.0,1.0,9643,"(32, 512, 512)",2.967549e-09,1.0,8388608,"(32, 512, 512)",0.0,1.0,267726
1,phe_0029,test,"(32, 512, 512)",0.0,1.0,152116,"(32, 512, 512)",0.0,1.0,72100,"(32, 512, 512)",3.411452e-07,1.0,8388608,"(32, 512, 512)",0.0,1.0,838090
2,phe_0033,test,"(32, 512, 512)",0.0,1.0,158663,"(32, 512, 512)",0.0,1.0,53226,"(32, 512, 512)",3.549697e-08,1.0,8388608,"(32, 512, 512)",0.0,1.0,726702
3,phe_0080,test,"(32, 512, 512)",0.0,1.0,134652,"(32, 512, 512)",0.0,1.0,47843,"(32, 512, 512)",1.156509e-08,1.0,8388608,"(32, 512, 512)",0.0,1.0,519246
4,phe_0084,test,"(32, 512, 512)",0.0,1.0,162551,"(32, 512, 512)",0.0,1.0,46880,"(32, 512, 512)",3.777170e-08,1.0,8388608,"(32, 512, 512)",0.0,1.0,599354
5,phe_0095,test,"(32, 512, 512)",0.0,1.0,67875,"(32, 512, 512)",0.0,1.0,13236,"(32, 512, 512)",1.871127e-10,1.0,8388608,"(32, 512, 512)",0.0,1.0,160251
6,phe_0096,test,"(30, 512, 512)",0.0,1.0,109570,"(30, 512, 512)",0.0,1.0,30425,"(30, 512, 512)",2.404009e-09,1.0,7864320,"(30, 512, 512)",0.0,1.0,464833
7,phe_0115,test,"(32, 512, 512)",0.0,1.0,126248,"(32, 512, 512)",0.0,1.0,45162,"(32, 512, 512)",1.835665e-08,1.0,8388608,"(32, 512, 512)",0.0,1.0,525760


## 3b. QC visualization for pseudo-ICH priors

PHE masks are shown only for visual sanity checking. They are not used to generate pseudo-ICH maps.


In [6]:
RUN_PRIOR_QC_FIGURES = True
N_PRIOR_QC_PER_BUCKET = 4

CT_WINDOWS = {
    'brain': (0, 80),
    'blood': (-50, 150),
    'wide': (-100, 300),
}

def window_ct(arr, low=-50, high=150):
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.clip(arr, low, high)
    return (arr - low) / max(high - low, 1e-6)

def largest_slice(mask):
    if not mask.any():
        return mask.shape[0] // 2
    areas = mask.reshape(mask.shape[0], -1).sum(axis=1)
    return int(np.argmax(areas))

def select_prior_qc_cases(prior_df, n_each=4):
    buckets = []
    buckets.append(prior_df.sort_values('binary_voxels').head(n_each).assign(qc_reason='small_prior'))
    buckets.append(prior_df.sort_values('binary_voxels', ascending=False).head(n_each).assign(qc_reason='large_prior'))
    test_df = prior_df[prior_df['split'] == 'test']
    if len(test_df):
        buckets.append(test_df.head(n_each).assign(qc_reason='test_cases'))
    out = pd.concat(buckets, ignore_index=True)
    return out.drop_duplicates('case_id', keep='first')

if RUN_PRIOR_QC_FIGURES:
    import matplotlib.pyplot as plt
    qc_df = select_prior_qc_cases(prior_df, N_PRIOR_QC_PER_BUCKET)
    qc_df.to_csv(PSEUDO_ICH_OUTPUT_ROOT / '02_19c_pseudo_prior_qc_selected_cases.csv', index=False)
    for row in qc_df.itertuples(index=False):
        case_id = str(row.case_id)
        split_row = split_df.loc[split_df['case_id'].astype(str) == case_id].iloc[0]
        ct = sitk.GetArrayFromImage(sitk.ReadImage(str(split_row.img_path)))
        phe = sitk.GetArrayFromImage(sitk.ReadImage(str(split_row.phe_mask_path))) > 0
        prob = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz')))
        binary = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'))) > 0
        distance = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz')))
        dilated = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz'))) > 0
        z = largest_slice(phe | binary | dilated)
        ct_blood = window_ct(ct[z], *CT_WINDOWS['blood'])
        ct_brain = window_ct(ct[z], *CT_WINDOWS['brain'])
        fig, axes = plt.subplots(1, 5, figsize=(18, 4))
        axes[0].imshow(ct_brain, cmap='gray')
        axes[0].set_title(f'{case_id}\n{row.qc_reason}\nCT brain')
        axes[1].imshow(ct_blood, cmap='gray')
        axes[1].imshow(np.ma.masked_where(~phe[z], phe[z]), cmap='Greens', alpha=0.55)
        axes[1].set_title('PHE GT')
        axes[2].imshow(ct_blood, cmap='gray')
        axes[2].imshow(prob[z], cmap='magma', alpha=0.60, vmin=0, vmax=1)
        axes[2].set_title('pseudo-ICH probability')
        axes[3].imshow(distance[z], cmap='viridis', vmin=0, vmax=1)
        axes[3].set_title('distance prior')
        axes[4].imshow(ct_blood, cmap='gray')
        axes[4].imshow(np.ma.masked_where(~dilated[z], dilated[z]), cmap='Blues', alpha=0.25)
        axes[4].imshow(np.ma.masked_where(~phe[z], phe[z]), cmap='Greens', alpha=0.45)
        axes[4].imshow(np.ma.masked_where(~binary[z], binary[z]), cmap='Reds', alpha=0.45)
        axes[4].set_title('PHE + pseudo-ICH + ROI')
        for ax in axes:
            ax.axis('off')
        fig.tight_layout()
        out = PRIOR_QC_DIR / f'{case_id}_{row.qc_reason}_prior_qc.png'
        fig.savefig(out, dpi=160, bbox_inches='tight')
        plt.close(fig)
    print('Saved pseudo-prior QC figures:', PRIOR_QC_DIR)
else:
    print('Skipped prior QC figures.')


Saved pseudo-prior QC figures: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\pseudo_ich_outputs\ct_prior_bw55otsu\visualizations


## 4. Convert PHE-SICH to selectable multi-channel nnU-Net raw dataset

The conversion is intentionally identical in split/labels to 02_12/02_14b, but the CT channels and prior channels are explicit. The default 19b-v2 input is a light multi-window stack: brain-window CT, subdural-window CT, pseudo-ICH probability, and distance prior. The optional triple-window experiment adds bone-window CT as a separate control.


In [7]:
OVERWRITE_RAW_DATASET = True

PRIOR_CHANNEL_PATHS = {
    'ich_binary': ICH_PRIOR_MASK_DIR,
    'ich_probability': ICH_PRIOR_PROB_DIR,
    'ich_distance': ICH_DISTANCE_PRIOR_DIR,
    'ich_dilated_roi': ICH_DILATED_ROI_DIR,
}
PRIOR_CHANNEL_DISPLAY_NAMES = {
    'ich_binary': 'ICH_binary_prior',
    'ich_probability': 'ICH_probability_prior',
    'ich_distance': 'ICH_distance_prior',
    'ich_dilated_roi': 'ICH_dilated_roi_prior',
}


def write_float_array_like_reference(arr, reference_img, dst_path: Path):
    out = sitk.GetImageFromArray(np.asarray(arr, dtype=np.float32))
    if out.GetSize() != reference_img.GetSize():
        raise ValueError(f'Size mismatch for {dst_path}: {out.GetSize()} vs ref {reference_img.GetSize()}')
    out.CopyInformation(reference_img)
    sitk.WriteImage(out, str(dst_path))


def write_ct_input_channels_like_reference(src_path: Path, image_dir: Path, case_id: str):
    ref = sitk.ReadImage(str(src_path))
    ct = sitk.GetArrayFromImage(ref)
    brain_mask = make_intracranial_mask(ct)
    stats = {
        'ct_preprocess_profile': CT_PREPROCESS_PROFILE,
        'ct_input_windows': '|'.join(CT_INPUT_WINDOW_NAMES),
        'brain_mask_voxels': int(brain_mask.sum()),
    }
    channel_record = {}
    for channel_offset, window_spec in enumerate(CT_INPUT_WINDOWS):
        ct_channel, _, window_stats = preprocess_ct_window_for_phe_input(ct, window_spec, brain_mask=brain_mask)
        dst_path = image_dir / f'{case_id}_{channel_offset:04d}.nii.gz'
        write_float_array_like_reference(ct_channel, ref, dst_path)
        name = str(window_spec['name'])
        channel_record[f'ct_{name}_channel'] = str(dst_path)
        stats.update(window_stats)
    return stats, channel_record


def write_mask_like_reference(src_path: Path, reference_path: Path, dst_path: Path):
    ref = sitk.ReadImage(str(reference_path))
    seg = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(seg)
    out = sitk.GetImageFromArray((arr > 0).astype(np.uint8))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))


def write_prior_like_reference(src_path: Path, reference_path: Path, dst_path: Path, prior_name: str):
    ref = sitk.ReadImage(str(reference_path))
    src = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(src)
    if prior_name in {'ich_binary', 'ich_dilated_roi'}:
        arr = (arr > 0).astype(np.uint8)
        dtype = np.uint8
    else:
        arr = np.clip(arr.astype(np.float32), 0.0, 1.0)
        dtype = np.float32
    out = sitk.GetImageFromArray(arr.astype(dtype))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))
    return float(arr.min()), float(arr.max()), int((arr > 0).sum())

if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    img_src = Path(row.img_path)
    phe_src = Path(row.phe_mask_path)

    if row.split == 'test':
        image_dir = imagesTs
        label_dir = labelsTs
    else:
        image_dir = imagesTr
        label_dir = labelsTr

    seg_dst = label_dir / f'{case_id}.nii.gz'
    ct_stats, ct_channel_record = write_ct_input_channels_like_reference(img_src, image_dir, case_id)

    channel_record = {}
    prior_channel_start = len(CT_INPUT_WINDOWS)
    for channel_offset, prior_name in enumerate(PRIOR_CHANNELS, start=prior_channel_start):
        prior_dir = PRIOR_CHANNEL_PATHS[prior_name]
        prior_src = prior_dir / f'{case_id}.nii.gz'
        assert prior_src.exists(), f'Missing {prior_name} for {case_id}: {prior_src}'
        prior_dst = image_dir / f'{case_id}_{channel_offset:04d}.nii.gz'
        prior_min, prior_max, prior_voxels = write_prior_like_reference(prior_src, img_src, prior_dst, prior_name)
        channel_record[f'{prior_name}_channel'] = str(prior_dst)
        channel_record[f'{prior_name}_min'] = prior_min
        channel_record[f'{prior_name}_max'] = prior_max
        channel_record[f'{prior_name}_voxels_gt0'] = prior_voxels

    write_mask_like_reference(phe_src, img_src, seg_dst)
    records.append({
        'case_id': case_id,
        'split': row.split,
        'phe_label': str(seg_dst),
        'source_image': str(img_src),
        'source_phe_mask': str(phe_src),
        **ct_stats,
        **ct_channel_record,
        **channel_record,
    })

channel_names = {}
for channel_offset, window_spec in enumerate(CT_INPUT_WINDOWS):
    name = str(window_spec['name'])
    channel_names[str(channel_offset)] = f"CT_{name}_L{int(window_spec['level'])}_W{int(window_spec['width'])}_{'skullstrip' if CT_SKULL_STRIP_ENABLED else 'noskullstrip'}"
for channel_offset, prior_name in enumerate(PRIOR_CHANNELS, start=len(CT_INPUT_WINDOWS)):
    channel_names[str(channel_offset)] = PRIOR_CHANNEL_DISPLAY_NAMES[prior_name]

dataset_json = {
    'channel_names': channel_names,
    'labels': {'background': 0, 'PHE': 1},
    'numTraining': int((split_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

conversion_df = pd.DataFrame(records)
conversion_manifest = OUTPUT_ROOT / f'02_19c_{EXPERIMENT_SHORT_NAME}_conversion_manifest.csv'
conversion_df.to_csv(conversion_manifest, index=False)

print('Experiment:', EXPERIMENT_ID)
print('CT input windows:', CT_INPUT_WINDOW_NAMES)
print('Prior channels:', PRIOR_CHANNELS)
print('Total input channels:', len(channel_names), channel_names)
print('Dataset dir:', DATASET_DIR)
print('imagesTr channel files:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs channel files:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
print('dataset.json:', DATASET_DIR / 'dataset.json')
print('conversion manifest:', conversion_manifest)
display(conversion_df.groupby('split').size().rename('cases').reset_index())
display(pd.DataFrame([dataset_json['channel_names']]).T.rename(columns={0: 'channel_name'}))


Experiment: phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval
CT input windows: ['brain', 'subdural']
Prior channels: ['ich_probability', 'ich_distance']
Total input channels: 4 {'0': 'CT_brain_L40_W120_skullstrip', '1': 'CT_subdural_L75_W215_skullstrip', '2': 'ICH_probability_prior', '3': 'ICH_distance_prior'}
Dataset dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_raw\Dataset198_PHESICH_PHE_19c_hardval_bw55
imagesTr channel files: 436
labelsTr: 109
imagesTs channel files: 44
labelsTs: 11
dataset.json: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_raw\Dataset198_PHESICH_PHE_19c_hardval_bw55\dataset.json
conversion manifest: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_19c_hardval_bw55otsu_conversion_manifest.csv


,split,cases
0,test,11
1,train,99
2,val,10


,channel_name
0,CT_brain_L40_W120_skullstrip
1,CT_subdural_L75_W215_skullstrip
2,ICH_probability_prior
3,ICH_distance_prior


## 5. Helper call entrypoints



In [8]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 6. Plan and preprocess selected Dataset



In [9]:
CONFIGURATION = '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped. Set RUN_PLAN_PREPROCESS=True to run.')


>>> nnUNetv2_plan_and_preprocess -d 198 -c 3d_fullres --verify_dataset_integrity -npfp 2 -np 1
Fingerprint extraction...
Dataset198_PHESICH_PHE_19c_hardval_bw55
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer


Extracting dataset fingerprint: 100%|██████████| 109/109 [00:27<00:00,  3.96it/s]


Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [5.         0.50292969 0.50292969]. 
Current patch size: (16, 320, 320). 
Current median shape: [ 30.         497.08737864 497.08737864]
Attempting to find 3d_lowres config. 
Current spacing: [5.         0.51801758 0.51801758]. 
Current patch size: (16, 320, 320). 
Current median shape: [ 30.         482.60910548 482.60910548]
Attempting to find 3d_lowres config. 
Current spacing: [5.         0.53355811 0.53355811]. 
Current patch size: (20, 320, 256). 
Current median shape: [ 30.         468.55252959 468.55252959]
Attempting to find 3d_lowres config. 
Current spacing: [5.         0.54956485 0.54956485]. 
Current patch size: (20, 32

Preprocessing cases: 100%|██████████| 109/109 [03:35<00:00,  1.98s/it]


Done in 4.5 min


## 7. Write manual fold-0 split



In [10]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = split_df.loc[split_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = split_df.loc[split_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)
    print('Run plan/preprocess first, then rerun this cell.')


Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_preprocessed\Dataset198_PHESICH_PHE_19c_hardval_bw55\splits_final.json
train: 99 val: 10 test: 11


## 8. Train PHE model with selected ICH prior channels

This trains a new PHE model for the current `EXPERIMENT_ID`. The pseudo-ICH remains frozen.


In [11]:
RUN_TRAIN = True
CONFIGURATION = globals().get('CONFIGURATION', '3d_fullres')
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FOLD_DIR.mkdir(parents=True, exist_ok=True)

EFFECTIVE_CONTINUE_TRAINING = CONTINUE_TRAINING
if CONTINUE_TRAINING:
    available_checkpoints = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth'))
    if not available_checkpoints:
        print('CONTINUE_TRAINING=True but no checkpoint exists yet; starting a fresh training run.')
        EFFECTIVE_CONTINUE_TRAINING = False
    else:
        print('Continuing from available checkpoints:', available_checkpoints)

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('Model dir:', MODEL_DIR)
    print('Fold dir :', FOLD_DIR)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    print('nnUNet_compile:', os.environ.get('nnUNet_compile'))
    print('continue_training:', EFFECTIVE_CONTINUE_TRAINING)
    # Windows/Jupyter guard: some nnU-Net builds try to copy dataset_fingerprint.json
    # before recreating the trainer output folder. Ensure the destination parent exists.
    import shutil as _shutil
    if not hasattr(_shutil, '_codex_original_copyfile'):
        _shutil._codex_original_copyfile = _shutil.copyfile

    def _copyfile_with_parent(src, dst, *args, **kwargs):
        if str(src).endswith('dataset_fingerprint.json'):
            Path(dst).parent.mkdir(parents=True, exist_ok=True)
        return _shutil._codex_original_copyfile(src, dst, *args, **kwargs)

    _shutil.copyfile = _copyfile_with_parent
    try:
        run_training(
            str(DATASET_ID),
            CONFIGURATION,
            str(FOLD),
            trainer_class_name=TRAINER,
            plans_identifier=PLANS,
            export_validation_probabilities=EXPORT_VAL_NPZ,
            continue_training=EFFECTIVE_CONTINUE_TRAINING,
            device=device,
        )
    finally:
        _shutil.copyfile = _shutil._codex_original_copyfile
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


CONTINUE_TRAINING=True but no checkpoint exists yet; starting a fresh training run.
Training device: cuda
Model dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\o19c_results\19c_hardval_bw55otsu\Dataset198_PHESICH_PHE_19c_hardval_bw55\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres
Fold dir : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\o19c_results\19c_hardval_bw55otsu\Dataset198_PHESICH_PHE_19c_hardval_bw55\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres\fold_0
nnUNet_n_proc_DA: 0
nnUNet_compile: False
continue_training: False

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & M

## 9. Predict test set and evaluate PHE



In [12]:
RUN_PHE_PREDICT = True
RUN_PHE_EVALUATE = True
DISABLE_TTA = False
CHECKPOINT = 'auto'

imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

print('Model dir        :', MODEL_DIR)
print('Fold dir         :', FOLD_DIR)
print('Using checkpoint :', RESOLVED_CHECKPOINT)

if RUN_PHE_PREDICT:
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('PHE predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=False,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction. Set RUN_PHE_PREDICT=True to run inference.')

if RUN_PHE_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped evaluation. Set RUN_PHE_EVALUATE=True after prediction.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


Model dir        : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\o19c_results\19c_hardval_bw55otsu\Dataset198_PHESICH_PHE_19c_hardval_bw55\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres
Fold dir         : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\o19c_results\19c_hardval_bw55otsu\Dataset198_PHESICH_PHE_19c_hardval_bw55\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres\fold_0
Using checkpoint : checkpoint_best.pth
PHE predict device: cuda
There are 11 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 11 cases that I would like to predict

Predicting phe_0002:
perform_everything_on_device: True


100%|██████████| 27/27 [00:13<00:00,  2.06it/s]


sending off prediction to background worker for resampling and export
done with phe_0002

Predicting phe_0029:
perform_everything_on_device: True


100%|██████████| 12/12 [00:05<00:00,  2.21it/s]


sending off prediction to background worker for resampling and export
done with phe_0029

Predicting phe_0033:
perform_everything_on_device: True


...

sending off prediction to background worker for resampling and export
done with phe_0130

Predicting phe_0138:
perform_everything_on_device: True


100%|██████████| 27/27 [00:13<00:00,  2.05it/s]


sending off prediction to background worker for resampling and export
done with phe_0138
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.
>>> nnUNetv2_evaluate_folder D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_raw\Dataset198_PHESICH_PHE_19c_hardval_bw55\labelsTs D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best -djfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_raw\Dataset198_PHESICH_PHE_19c_hardval_bw55\dataset.json -pfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_preprocessed\Dataset198_PHESICH_PHE_19c_hardval_bw55\nnUNetPlans.json -o D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\summary.json -np 2
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer


Done in 0.0 min
Prediction dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best
Summary json  : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\summary.json


## 9b. Pseudo-ICH-guided post-processing


In [13]:
RUN_POSTPROCESS = True
POSTPROCESS_MIN_SIZE_VOXELS = 50
POSTPROCESS_ROI_RADIUS_MM = 25.0
POSTPROCESS_USE_ICH_ROI_FILTER = False  # keep False by default; the old ICH-distance filter removed many small PHE cases.

POSTPROCESS_PRED_DIR = OUTPUT_ROOT / f'postprocessed_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
POSTPROCESS_SUMMARY_JSON = POSTPROCESS_PRED_DIR / 'summary.json'


def make_ich_roi(binary_ich, spacing_xyz, radius_mm):
    binary_ich = binary_ich.astype(bool)
    if not binary_ich.any():
        return np.ones(binary_ich.shape, dtype=bool)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return dist_mm <= float(radius_mm)


def postprocess_phe_light(phe_pred, ich_prior=None, spacing_xyz=None, min_size=50, radius_mm=25.0, use_ich_roi=False):
    phe_pred = phe_pred.astype(bool)
    if use_ich_roi and ich_prior is not None and spacing_xyz is not None:
        ich_roi = make_ich_roi(ich_prior.astype(bool), spacing_xyz, radius_mm)
    else:
        ich_roi = np.ones(phe_pred.shape, dtype=bool)
    labeled, num = ndi.label(phe_pred)
    output = np.zeros_like(phe_pred, dtype=np.uint8)
    kept = 0
    removed_small = 0
    removed_far = 0
    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        comp_size = int(comp.sum())
        if comp_size < int(min_size):
            removed_small += 1
            continue
        if use_ich_roi and np.logical_and(comp, ich_roi).sum() == 0:
            removed_far += 1
            continue
        output[comp] = 1
        kept += 1
    return output, {
        'components': int(num),
        'kept': int(kept),
        'removed_small': int(removed_small),
        'removed_far': int(removed_far),
        'use_ich_roi_filter': bool(use_ich_roi),
    }


def write_mask_array_like_reference(mask_arr, reference_path: Path, out_path: Path):
    ref = sitk.ReadImage(str(reference_path))
    out = sitk.GetImageFromArray(mask_arr.astype(np.uint8))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))


if RUN_POSTPROCESS:
    POSTPROCESS_PRED_DIR.mkdir(parents=True, exist_ok=True)
    pp_rows = []
    for case_id in split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str):
        raw_pred_path = PRED_DIR / f'{case_id}.nii.gz'
        ich_prior_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
        out_path = POSTPROCESS_PRED_DIR / f'{case_id}.nii.gz'
        if not raw_pred_path.exists():
            print('Missing raw PHE prediction:', raw_pred_path)
            continue
        raw_img = sitk.ReadImage(str(raw_pred_path))
        raw_pred = sitk.GetArrayFromImage(raw_img) > 0
        ich_prior = sitk.GetArrayFromImage(sitk.ReadImage(str(ich_prior_path))) > 0 if ich_prior_path.exists() else None
        spacing_xyz = np.array(raw_img.GetSpacing(), dtype=float)
        pp_pred, stats = postprocess_phe_light(
            raw_pred,
            ich_prior=ich_prior,
            spacing_xyz=spacing_xyz,
            min_size=POSTPROCESS_MIN_SIZE_VOXELS,
            radius_mm=POSTPROCESS_ROI_RADIUS_MM,
            use_ich_roi=POSTPROCESS_USE_ICH_ROI_FILTER,
        )
        write_mask_array_like_reference(pp_pred, raw_pred_path, out_path)
        pp_rows.append({
            'case_id': case_id,
            'raw_voxels': int(raw_pred.sum()),
            'post_voxels': int(pp_pred.sum()),
            **stats,
        })
    postprocess_df = pd.DataFrame(pp_rows)
    postprocess_manifest = OUTPUT_ROOT / f'02_19c_{EXPERIMENT_ID}_postprocess_manifest.csv'
    postprocess_df.to_csv(postprocess_manifest, index=False)
    print('Saved postprocessed predictions:', POSTPROCESS_PRED_DIR)
    print('Saved postprocess manifest:', postprocess_manifest)
    print('Postprocess uses ICH ROI filter:', POSTPROCESS_USE_ICH_ROI_FILTER)
    display(postprocess_df)

    if RUN_PHE_EVALUATE and len(postprocess_df):
        plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
        dataset_json_file = DATASET_DIR / 'dataset.json'
        call_entrypoint(evaluate_folder_entry_point, [
            'nnUNetv2_evaluate_folder',
            str(labelsTs),
            str(POSTPROCESS_PRED_DIR),
            '-djfile', str(dataset_json_file),
            '-pfile', str(plans_file),
            '-o', str(POSTPROCESS_SUMMARY_JSON),
            '-np', '2',
        ])
else:
    print('Skipped light PHE post-processing.')

Saved postprocessed predictions: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\postprocessed_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best
Saved postprocess manifest: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval_postprocess_manifest.csv
Postprocess uses ICH ROI filter: False


,case_id,raw_voxels,post_voxels,components,kept,removed_small,removed_far,use_ich_roi_filter
0,phe_0002,577,564,4,1,3,0,False
1,phe_0029,7747,7709,9,7,2,0,False
2,phe_0033,1653,1585,8,3,5,0,False
3,phe_0080,1107,1098,2,1,1,0,False
4,phe_0084,1523,1509,2,1,1,0,False
5,phe_0095,133,79,4,1,3,0,False
6,phe_0096,5682,5558,12,5,7,0,False
7,phe_0115,1674,1659,7,5,2,0,False
8,phe_0119,1515,1480,10,4,6,0,False
9,phe_0130,3984,3881,9,3,6,0,False


>>> nnUNetv2_evaluate_folder D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_raw\Dataset198_PHESICH_PHE_19c_hardval_bw55\labelsTs D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\postprocessed_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best -djfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_raw\Dataset198_PHESICH_PHE_19c_hardval_bw55\dataset.json -pfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\nnunet_workdir\nnUNet_preprocessed\Dataset198_PHESICH_PHE_19c_hardval_bw55\nnUNetPlans.json -o D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\postprocessed_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\summary.json -np 2
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Done in 0.0 min


## 10. Export compact metrics CSV


In [14]:
def strip_nii_gz_name(path_like):
    name = Path(str(path_like)).name
    return name[:-7] if name.endswith('.nii.gz') else Path(name).stem

def export_summary(summary_json: Path, source_name: str):
    if not summary_json.exists():
        print('No summary yet:', summary_json)
        return None, None
    with open(summary_json, 'r', encoding='utf-8') as f:
        summary = json.load(f)

    rows = []
    for item in summary.get('metric_per_case', []):
        metrics = item.get('metrics', {}).get('1', {})
        rows.append({
            'experiment_id': EXPERIMENT_ID,
            'source': source_name,
            'case_id': strip_nii_gz_name(item.get('prediction_file', '')),
            'prediction_file': item.get('prediction_file', ''),
            'reference_file': item.get('reference_file', ''),
            **metrics,
        })
    per_case_df = pd.DataFrame(rows)

    summary_rows = []
    numeric_cols = [c for c in per_case_df.columns if c not in {'experiment_id', 'source', 'case_id', 'prediction_file', 'reference_file'}]
    for col in numeric_cols:
        vals = pd.to_numeric(per_case_df[col], errors='coerce').dropna()
        if len(vals) == 0:
            continue
        summary_rows.append({
            'experiment_id': EXPERIMENT_ID,
            'source': source_name,
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    return per_case_df, summary_df

raw_per_case_df, raw_summary_df = export_summary(SUMMARY_JSON, 'raw')
post_per_case_df, post_summary_df = export_summary(globals().get('POSTPROCESS_SUMMARY_JSON', Path('__missing__')), 'postprocessed')

per_case_frames = [df for df in [raw_per_case_df, post_per_case_df] if df is not None]
summary_frames = [df for df in [raw_summary_df, post_summary_df] if df is not None]

if per_case_frames:
    metrics_df = pd.concat(per_case_frames, ignore_index=True)
    metrics_csv = OUTPUT_ROOT / f'02_19c_{EXPERIMENT_SHORT_NAME}_metrics_per_case_raw_vs_post.csv'
    metrics_df.to_csv(metrics_csv, index=False)
    print('Wrote:', metrics_csv)
else:
    print('No per-case metrics exported.')

if summary_frames:
    summary_df = pd.concat(summary_frames, ignore_index=True)
    summary_csv = OUTPUT_ROOT / f'02_19c_{EXPERIMENT_SHORT_NAME}_metrics_summary_raw_vs_post.csv'
    summary_df.to_csv(summary_csv, index=False)
    print('Wrote:', summary_csv)
    display(summary_df.pivot_table(index='metric', columns='source', values='mean'))
else:
    print('No metric summary exported. Run train -> predict -> evaluate first.')

experiment_record = {
    'notebook_id': '02_19c',
    'experiment_id': EXPERIMENT_ID,
    'experiment_short_name': EXPERIMENT_SHORT_NAME,
    'dataset_id': DATASET_ID,
    'dataset_name': DATASET_NAME,
    'prior_channels': '|'.join(PRIOR_CHANNELS),
    'prior_source': PSEUDO_ICH_SOURCE,
    'prior_policy': PSEUDO_ICH_POLICY,
    'ct_input_windows': '|'.join(CT_INPUT_WINDOW_NAMES),
    'pseudo_soft_low_hu': PSEUDO_ICH_SOFT_LOW_HU,
    'pseudo_fixed_threshold_hu': PSEUDO_ICH_FIXED_THRESHOLD_HU,
    'pseudo_core_low_hu': PSEUDO_ICH_CORE_LOW_HU,
    'pseudo_core_high_hu': PSEUDO_ICH_CORE_HIGH_HU,
    'pseudo_soft_high_hu': PSEUDO_ICH_SOFT_HIGH_HU,
    'pseudo_local_otsu_enabled': PSEUDO_ICH_LOCAL_OTSU_ENABLED,
    'pseudo_local_otsu_roi_dilate_mm': PSEUDO_ICH_LOCAL_OTSU_DILATE_RADIUS_MM,
    'pseudo_local_otsu_clip_low_hu': PSEUDO_ICH_LOCAL_OTSU_CLIP_LOW_HU,
    'pseudo_local_otsu_clip_high_hu': PSEUDO_ICH_LOCAL_OTSU_CLIP_HIGH_HU,
    'pseudo_high_hu_artifact_cutoff': PSEUDO_ICH_HIGH_HU_ARTIFACT_CUTOFF,
    'pseudo_high_hu_adj_radius_mm': PSEUDO_ICH_HIGH_HU_ADJ_RADIUS_MM,
    'pseudo_morph_closing_kernel_px': PSEUDO_ICH_MORPH_CLOSING_KERNEL_PX,
    'pseudo_morph_opening_kernel_px': PSEUDO_ICH_MORPH_OPENING_KERNEL_PX,
    'ct_preprocess_profile': CT_PREPROCESS_PROFILE,
    'ct_window_low_hu': CT_WINDOW_LOW_HU,
    'ct_window_high_hu': CT_WINDOW_HIGH_HU,
    'ct_skull_strip_enabled': CT_SKULL_STRIP_ENABLED,
    'postprocess_min_size_voxels': globals().get('POSTPROCESS_MIN_SIZE_VOXELS', None),
    'postprocess_use_ich_roi_filter': globals().get('POSTPROCESS_USE_ICH_ROI_FILTER', None),
    'output_root': str(OUTPUT_ROOT),
}
experiment_summary_csv = OUTPUT_BASE / '02_19c_experiment_registry.csv'
existing = pd.read_csv(experiment_summary_csv) if experiment_summary_csv.exists() else pd.DataFrame()
registry_df = pd.concat([existing, pd.DataFrame([experiment_record])], ignore_index=True)
registry_df = registry_df.drop_duplicates(['experiment_id', 'dataset_id'], keep='last')
registry_df.to_csv(experiment_summary_csv, index=False)
print('Updated 02_19 experiment registry:', experiment_summary_csv)


Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_19c_hardval_bw55otsu_metrics_per_case_raw_vs_post.csv
Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_19c_hardval_bw55otsu_metrics_summary_raw_vs_post.csv


source,postprocessed,raw
metric,,
Dice,3.989490e-01,4.026815e-01
FN,1.635364e+03,1.616818e+03
FP,1.189455e+03,1.213909e+03
IoU,2.688120e-01,2.712463e-01
TN,7.955599e+06,7.955574e+06
TP,1.221364e+03,1.239909e+03
n_pred,2.410818e+03,2.453818e+03
n_ref,2.856727e+03,2.856727e+03


Updated 02_19 experiment registry: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\02_19c_experiment_registry.csv


## 11. PHE detection / early detection analysis


In [15]:
RUN_PHE_DETECTION_ANALYSIS = True

# GT positive threshold should usually stay at 0: any manual PHE voxel means PHE is present.
GT_POSITIVE_VOLUME_THRESHOLD_ML = 0.0

# Main operating point. The threshold sweep below helps decide whether this should change.
PRED_POSITIVE_VOLUME_THRESHOLD_ML = 0.5
DETECTION_THRESHOLD_SWEEP_ML = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]

# Proxy for early/subtle PHE when onset-time metadata is unavailable.
SMALL_PHE_VOLUME_CUTOFF_ML = 5.0


def load_binary_mask(path: Path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img) > 0
    return img, arr


def mask_volume_ml(mask, spacing_xyz):
    voxel_ml = float(np.prod(np.asarray(spacing_xyz, dtype=float)) / 1000.0)
    return float(mask.sum() * voxel_ml)


def safe_div(num, den):
    return np.nan if den == 0 else float(num / den)


def detection_counts(gt_positive, pred_positive):
    gt_positive = np.asarray(gt_positive, dtype=bool)
    pred_positive = np.asarray(pred_positive, dtype=bool)
    tp = int(np.logical_and(gt_positive, pred_positive).sum())
    fp = int(np.logical_and(~gt_positive, pred_positive).sum())
    fn = int(np.logical_and(gt_positive, ~pred_positive).sum())
    tn = int(np.logical_and(~gt_positive, ~pred_positive).sum())
    return tp, fp, fn, tn


def detection_metric_row(df, source, cohort, pred_threshold_ml):
    tp, fp, fn, tn = detection_counts(df['gt_positive'], df['pred_positive'])
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    precision = safe_div(tp, tp + fp)
    recall = sensitivity
    npv = safe_div(tn, tn + fn)
    f1 = np.nan if precision != precision or recall != recall or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    accuracy = safe_div(tp + tn, tp + fp + fn + tn)
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    return {
        'experiment_id': EXPERIMENT_ID,
        'source': source,
        'cohort': cohort,
        'pred_threshold_ml': float(pred_threshold_ml),
        'gt_threshold_ml': float(GT_POSITIVE_VOLUME_THRESHOLD_ML),
        'small_phe_cutoff_ml': float(SMALL_PHE_VOLUME_CUTOFF_ML),
        'n': int(len(df)),
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'recall': recall,
        'npv': npv,
        'f1': f1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'gt_positive_rate': float(pd.to_numeric(df['gt_positive'], errors='coerce').mean()) if len(df) else np.nan,
        'pred_positive_rate': float(pd.to_numeric(df['pred_positive'], errors='coerce').mean()) if len(df) else np.nan,
        'mean_gt_volume_ml': float(df['gt_volume_ml'].mean()) if len(df) else np.nan,
        'mean_pred_volume_ml': float(df['pred_volume_ml'].mean()) if len(df) else np.nan,
    }


def available_detection_sources():
    sources = []
    if 'PRED_DIR' in globals():
        sources.append(('raw', Path(PRED_DIR)))
    if 'POSTPROCESS_PRED_DIR' in globals() and Path(POSTPROCESS_PRED_DIR).exists():
        sources.append(('postprocessed', Path(POSTPROCESS_PRED_DIR)))
    return sources


if RUN_PHE_DETECTION_ANALYSIS:
    if 'labelsTs' not in globals():
        raise RuntimeError('labelsTs is not defined. Run dataset conversion/prediction cells before this detection analysis.')

    detection_sources = available_detection_sources()
    if not detection_sources:
        print('No prediction folders found. Run PHE prediction first, then rerun this cell.')
    else:
        case_rows = []
        test_case_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()
        for source_name, pred_dir in detection_sources:
            for case_id in test_case_ids:
                gt_path = labelsTs / f'{case_id}.nii.gz'
                pred_path = pred_dir / f'{case_id}.nii.gz'
                row = {
                    'experiment_id': EXPERIMENT_ID,
                    'source': source_name,
                    'case_id': case_id,
                    'gt_path': str(gt_path),
                    'pred_path': str(pred_path),
                    'status': 'ok',
                }
                if not gt_path.exists():
                    row['status'] = 'missing_gt'
                    case_rows.append(row)
                    continue
                if not pred_path.exists():
                    row['status'] = 'missing_prediction'
                    case_rows.append(row)
                    continue

                gt_img, gt_mask = load_binary_mask(gt_path)
                _, pred_mask = load_binary_mask(pred_path)
                if gt_mask.shape != pred_mask.shape:
                    row['status'] = f'shape_mismatch_gt_{gt_mask.shape}_pred_{pred_mask.shape}'
                    case_rows.append(row)
                    continue

                spacing_xyz = np.array(gt_img.GetSpacing(), dtype=float)
                gt_volume_ml = mask_volume_ml(gt_mask, spacing_xyz)
                pred_volume_ml = mask_volume_ml(pred_mask, spacing_xyz)
                gt_positive = gt_volume_ml > GT_POSITIVE_VOLUME_THRESHOLD_ML
                pred_positive = pred_volume_ml >= PRED_POSITIVE_VOLUME_THRESHOLD_ML
                row.update({
                    'gt_volume_ml': gt_volume_ml,
                    'pred_volume_ml': pred_volume_ml,
                    'gt_voxels': int(gt_mask.sum()),
                    'pred_voxels': int(pred_mask.sum()),
                    'gt_positive': bool(gt_positive),
                    'pred_positive': bool(pred_positive),
                    'small_phe_proxy': bool(gt_positive and gt_volume_ml <= SMALL_PHE_VOLUME_CUTOFF_ML),
                    'volume_error_ml': float(pred_volume_ml - gt_volume_ml),
                    'abs_volume_error_ml': float(abs(pred_volume_ml - gt_volume_ml)),
                })
                case_rows.append(row)

        detection_case_df = pd.DataFrame(case_rows)
        detection_cases_csv = OUTPUT_ROOT / f'02_19c_{EXPERIMENT_ID}_phe_detection_per_case.csv'
        detection_case_df.to_csv(detection_cases_csv, index=False)
        print('Wrote:', detection_cases_csv)

        valid_df = detection_case_df[detection_case_df['status'] == 'ok'].copy()
        summary_rows = []
        threshold_rows = []
        if len(valid_df):
            for source_name, source_df in valid_df.groupby('source'):
                gt_class_counts = source_df['gt_positive'].value_counts(dropna=False).to_dict()
                print(f'GT detection class counts for {source_name}:', gt_class_counts)
                if source_df['gt_positive'].nunique(dropna=True) < 2:
                    print(f'Warning: {source_name} has only one GT detection class; specificity and balanced accuracy may be limited.')
                summary_rows.append(detection_metric_row(source_df, source_name, 'all_test', PRED_POSITIVE_VOLUME_THRESHOLD_ML))

                small_df = source_df[source_df['small_phe_proxy']]
                if len(small_df):
                    summary_rows.append(detection_metric_row(
                        small_df,
                        source_name,
                        f'small_phe_le_{SMALL_PHE_VOLUME_CUTOFF_ML:g}ml',
                        PRED_POSITIVE_VOLUME_THRESHOLD_ML,
                    ))

                large_df = source_df[source_df['gt_positive'] & ~source_df['small_phe_proxy']]
                if len(large_df):
                    summary_rows.append(detection_metric_row(
                        large_df,
                        source_name,
                        f'larger_phe_gt_{SMALL_PHE_VOLUME_CUTOFF_ML:g}ml',
                        PRED_POSITIVE_VOLUME_THRESHOLD_ML,
                    ))

                for threshold_ml in DETECTION_THRESHOLD_SWEEP_ML:
                    sweep_df = source_df.copy()
                    sweep_df['pred_positive'] = sweep_df['pred_volume_ml'] >= float(threshold_ml)
                    threshold_rows.append(detection_metric_row(sweep_df, source_name, 'threshold_sweep_all_test', threshold_ml))

            detection_summary_df = pd.DataFrame(summary_rows)
            detection_threshold_sweep_df = pd.DataFrame(threshold_rows)
            detection_summary_csv = OUTPUT_ROOT / f'02_19c_{EXPERIMENT_ID}_phe_detection_summary.csv'
            detection_threshold_sweep_csv = OUTPUT_ROOT / f'02_19c_{EXPERIMENT_ID}_phe_detection_threshold_sweep.csv'
            detection_summary_df.to_csv(detection_summary_csv, index=False)
            detection_threshold_sweep_df.to_csv(detection_threshold_sweep_csv, index=False)
            print('Wrote:', detection_summary_csv)
            print('Wrote:', detection_threshold_sweep_csv)
            display(detection_summary_df[['source', 'cohort', 'n', 'tp', 'fp', 'fn', 'tn', 'sensitivity', 'specificity', 'precision', 'f1', 'accuracy']])
            display(detection_threshold_sweep_df[['source', 'pred_threshold_ml', 'sensitivity', 'specificity', 'precision', 'f1', 'accuracy']])
            display(valid_df.sort_values(['source', 'gt_volume_ml'])[['source', 'case_id', 'gt_volume_ml', 'pred_volume_ml', 'gt_positive', 'pred_positive', 'small_phe_proxy']])
        else:
            print('No valid prediction/GT pairs found for detection analysis.')
else:
    print('Skipped PHE detection analysis.')


Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval_phe_detection_per_case.csv
GT detection class counts for postprocessed: {True: 11}
GT detection class counts for raw: {True: 11}
Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval_phe_detection_summary.csv
Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_19c\19c_hardval_bw55otsu\02_19c_phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval_phe_detection_threshold_sweep.csv


,source,cohort,n,tp,fp,fn,tn,sensitivity,specificity,precision,f1,accuracy
0,postprocessed,all_test,11,10,0,1,0,0.909091,NaN,1.0,0.952381,0.909091
1,postprocessed,small_phe_le_5ml,8,8,0,0,0,1.000000,NaN,1.0,1.000000,1.000000
2,postprocessed,larger_phe_gt_5ml,3,2,0,1,0,0.666667,NaN,1.0,0.800000,0.666667
3,raw,all_test,11,10,0,1,0,0.909091,NaN,1.0,0.952381,0.909091
4,raw,small_phe_le_5ml,8,8,0,0,0,1.000000,NaN,1.0,1.000000,1.000000
5,raw,larger_phe_gt_5ml,3,2,0,1,0,0.666667,NaN,1.0,0.800000,0.666667


,source,pred_threshold_ml,sensitivity,specificity,precision,f1,accuracy
0,postprocessed,0.00,1.000000,NaN,1.0,1.000000,1.000000
1,postprocessed,0.10,1.000000,NaN,1.0,1.000000,1.000000
2,postprocessed,0.25,0.909091,NaN,1.0,0.952381,0.909091
3,postprocessed,0.50,0.909091,NaN,1.0,0.952381,0.909091
4,postprocessed,1.00,0.818182,NaN,1.0,0.900000,0.818182
5,postprocessed,2.00,0.363636,NaN,1.0,0.533333,0.363636
6,raw,0.00,1.000000,NaN,1.0,1.000000,1.000000
7,raw,0.10,1.000000,NaN,1.0,1.000000,1.000000
8,raw,0.25,0.909091,NaN,1.0,0.952381,0.909091
9,raw,0.50,0.909091,NaN,1.0,0.952381,0.909091


,source,case_id,gt_volume_ml,pred_volume_ml,gt_positive,pred_positive,small_phe_proxy
18,postprocessed,phe_0115,0.620159,2.222124,True,True,True
20,postprocessed,phe_0130,1.555681,4.626513,True,True,True
14,postprocessed,phe_0080,1.659393,1.308918,True,True,True
11,postprocessed,phe_0002,1.949072,0.672340,True,True,True
19,postprocessed,phe_0119,2.287626,1.764297,True,True,True
21,postprocessed,phe_0138,2.591610,1.665354,True,True,True
15,postprocessed,phe_0084,2.751509,1.522562,True,True,True
12,postprocessed,phe_0029,3.897568,7.116615,True,True,True
16,postprocessed,phe_0095,5.660169,0.123251,True,False,False
13,postprocessed,phe_0033,6.607771,1.889467,True,True,False


## Notes

- `02_19c` uses a locked resplit for secondary comparison. Keep it separate from the original benchmark split unless all compared models are rerun on this exact split.
- Split counts remain `train=99`, `val=10`, `test=11`. The final IDs are defined directly in the setup cell as `VAL_SCAN_IDS` and `TEST_SCAN_IDS`.
- The default experiment is `phe_pseudo_prob_distance_prior_brain_subdural_ct55_otsu_resplit_hardval`, writing to `outputs_02_19c/19c_hardval_bw55otsu`, `Dataset198_PHESICH_PHE_19c_hardval_bw55`, and `o19c_results/19c_hardval_bw55otsu`.
- CT input remains the light multi-window stack: brain window L=40/W=120 + subdural window L=75/W=215 + pseudo-ICH probability + distance prior.
- Pseudo-ICH generation is still CT-only: fixed 55 HU global threshold, local Otsu inside a CT-derived high-density ROI, >80 HU artifact/calcification control by adjacency to the 40-80 HU core, and morphology/component filtering.
- Post-processing is conservative by default: remove connected components smaller than 50 voxels only. The older ICH-distance ROI filter is disabled because it removed many small PHE cases.
- PHE masks are used only as PHE segmentation labels/evaluation targets and optional QC overlays, not to generate prior channels.
